In [12]:
import json
import os
import tempfile
from docx import Document
from google.cloud import storage

In [13]:
def parse_gcs_uri(uri):
    """Estrae bucket_name e blob_name da un URI gs://"""
    if not uri.startswith("gs://"):
        return None, None
    parts = uri[5:].split("/", 1)
    bucket_name = parts[0]
    blob_name = parts[1] if len(parts) > 1 else ""
    return bucket_name, blob_name

def download_from_gcs(gcs_uri, local_path):
    """Scarica un file da GCS al percorso locale specificato."""
    bucket_name, blob_name = parse_gcs_uri(gcs_uri)
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.download_to_filename(local_path)
    print(f"Scaricato: {gcs_uri} -> {local_path}")

def upload_to_gcs(local_path, gcs_uri):
    """Carica un file locale su GCS."""
    bucket_name, blob_name = parse_gcs_uri(gcs_uri)
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(local_path)
    print(f"Caricato: {local_path} -> {gcs_uri}")

In [24]:
def is_heading_level(style_name, level):
    #Fa il check se lo stile corrisponde al livello che stiamo per scrivere (True o False)
    style_name = style_name.lower()
    target_en = f"heading {level}"
    target_it = f"titolo {level}"
    return target_en in style_name or target_it in style_name

def converti_docx_in_jsonl(input_path, output_path):

    temp_input = None
    temp_output = None
    
    # Se l'input è su GCS, scaricalo in un file temporaneo
    real_input_path = input_path
    if input_path.startswith("gs://"):
        temp_input = tempfile.NamedTemporaryFile(delete=False, suffix=".docx")
        temp_input.close() # Chiudiamo per permettere a download_blob di scriverci
        download_from_gcs(input_path, temp_input.name)
        real_input_path = temp_input.name

        print(real_input_path)

    # Determina il percorso di output reale (locale)
    real_output_path = output_path
    if output_path.startswith("gs://"):
        temp_output = tempfile.NamedTemporaryFile(delete=False, suffix=".jsonl")
        temp_output.close()
        real_output_path = temp_output.name
    
    doc = Document(real_input_path)
    print(f"\n--- INIZIO ELABORAZIONE: {input_path} ---\n")
    
    current_section = None      # Corrisponde a Heading 1
    current_paragraph = None    # Corrisponde a Heading 2
    buffer_text = []            # Accumulatore per il testo (Opzione B: raggruppamento)

    # Definisco una funzione internamente per scrivere nel file JSONL
    # La definisco qui perchè tanto viene usata solo qui dentro e legge direttamente le variabili definite sopra
    def flush_buffer(f_out):
        
        if buffer_text and current_section: # deve esserci sia del testo da scrivere che una sezione definita
            print(f"   >>> [SCRITTURA SU DISCO] Scrivo JSONL")
            print(f"       Sezione:   '{current_section}'")
            print(f"       Paragrafo: '{current_paragraph}'")
            print(f"       Testo ({len(full_text)} chars): '{full_text[:50]}...'")
            full_text = " ".join(buffer_text).strip()
            if full_text: # non metto righe con testo vuoto
                entry = {
                    "SEZIONE": current_section,
                    "PARAGRAFO": current_paragraph if current_paragraph else current_section,
                    "testo": full_text
                }
                f_out.write(json.dumps(entry, ensure_ascii=False) + '\n')
        
        buffer_text.clear()

    with open(real_output_path, 'w', encoding='utf-8') as f_out:
        
        # Considero solo i paragrafi (le tabelle sono escluse automaticamente da doc.paragraphs)
        for paragraph in doc.paragraphs:
            text = paragraph.text.strip()
            style_name = paragraph.style.name
            
            # sorvolo paragrafo vuoto
            if not text:
                continue

            # CASO NUOVA SEZIONE (Heading 1)
            
            if is_heading_level(style_name, 1):
                print(f"   -> Rilevato TITOLO 1 (Cambio Sezione)")
                # 1.Salvo tutto quello che ho
                flush_buffer(f_out)
                
                # 2. Aggiorno tutto 
                current_section = text
                current_paragraph = text 

                print(f"   -> Stato impostato: SEZIONE='{current_section}'")
            
            # CASO NUOVO PARAGRAFO (Heading 2)
            elif is_heading_level(style_name, 2):
                print(f"   -> Rilevato TITOLO 2 (Cambio Paragrafo)")
                # 1. Salvo tutto il paragrafo precedente
                flush_buffer(f_out)
                
                # 2. Aggiorno solo il paragrafo corrente
                current_paragraph = text
                print(f"   -> Stato impostato: PARAGRAFO='{current_paragraph}'")
            
            # CASO Testo normale o livelli più profondi (Heading 3+)
            else:
                print(f"   -> Aggiungo al buffer corrente")
                # 1. Aggiungo il testo nel buffer corrente
                buffer_text.append(text)

        # Alla fine del ciclo, salvo l'ultimo blocco di testo rimasto nel buffer
        print(f"\n[FINE FILE] Eseguo flush finale.")
        flush_buffer(f_out)
        
        if output_path.startswith("gs://"):
            upload_to_gcs(real_output_path, output_path)

        if temp_input and os.path.exists(temp_input.name):
            os.remove(temp_input.name)
        if temp_output and os.path.exists(temp_output.name):
            os.remove(temp_output.name)

    print(f"Operazione completata con successo.")



In [25]:
input_file = "gs://data-platform-framework-genai/Demo_Unipol_Nota_Integrativa/Test creazione file JSONL da DOCX/Passivo - Patrimonio Netto TEST - no dati.docx"
output_file = "gs://data-platform-framework-genai/Demo_Unipol_Nota_Integrativa/Test creazione file JSONL da DOCX/Patrimonio_Netto.jsonl"
converti_docx_in_jsonl(input_file, output_file)



Scaricato: gs://data-platform-framework-genai/Demo_Unipol_Nota_Integrativa/Test creazione file JSONL da DOCX/Passivo - Patrimonio Netto TEST - no dati.docx -> /var/tmp/tmp7e413azk.docx
/var/tmp/tmp7e413azk.docx

--- INIZIO ELABORAZIONE: gs://data-platform-framework-genai/Demo_Unipol_Nota_Integrativa/Test creazione file JSONL da DOCX/Passivo - Patrimonio Netto TEST - no dati.docx ---

   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
   -> Aggiungo al buffer corrente
 